In [1]:
import os
import sys
import oracledb

# ==============================================================================
# ORACLE CLIENT INITIALIZATION (DYNAMIC & VENV COMPATIBLE)
# ==============================================================================
DEFAULT_CLIENT_DIR = r"C:\Users\E1445612\oracle64client"
INSTANT_CLIENT_DIR = os.environ.get("ORACLE_HOME", DEFAULT_CLIENT_DIR)

if sys.platform.startswith("win32") and os.path.exists(INSTANT_CLIENT_DIR):
    try:
        os.add_dll_directory(INSTANT_CLIENT_DIR)
    except Exception as dll_err:
        print(f"[-] Warning adding DLL directory: {dll_err}")

try:
    oracledb.init_oracle_client(lib_dir=INSTANT_CLIENT_DIR)
    print(f"[+] Oracle Client initialized in Thick mode from: {INSTANT_CLIENT_DIR}")
except Exception as e:
    print(f"[-] Failed to initialize Oracle Client: {e}")
    sys.exit(1)

# Database connection credentials
DB_USER = "apps"
DB_PASS = "Sukh#na7"
DB_DSN = "BU2EBS.EMRSN.COM:34101/BETSYU2"


def fetch_sla_errors_fast():
    print("\n" + "=" * 80)
    print(" ULTRA-FAST FETCH: SLA ERRORS FOR INSTANCE CLONE DATE")
    print("=" * 80)

    print(f"[+] Connecting to Oracle EBS Database...")
    try:
        connection = oracledb.connect(
            user=DB_USER, 
            password=DB_PASS, 
            dsn=DB_DSN,
            tcp_connect_timeout=10.0
        )
        cursor = connection.cursor()
        print("[+] Connected successfully to EBS Instance.\n")
    except Exception as exc:
        print(f"[-] Database Connection Failed: {exc}")
        return

    clone_date = None

    # Step 1: Query Clone/Creation date from v$database or fnd_product_groups
    try:
        print("[+] Checking Instance Clone Date from v$database...")
        cursor.execute("SELECT TRUNC(created) FROM v$database")
        row = cursor.fetchone()
        if row and row[0]:
            clone_date = row[0]
    except Exception:
        pass

    if not clone_date:
        try:
            print("[+] Checking Instance Clone Date from fnd_product_groups...")
            cursor.execute("SELECT TRUNC(creation_date) FROM apps.fnd_product_groups")
            row = cursor.fetchone()
            if row and row[0]:
                clone_date = row[0]
        except Exception as err:
            print(f"[-] Error fetching clone date: {err}")

    if not clone_date:
        print("[-] Could not determine instance clone date. Exiting.")
        cursor.close()
        connection.close()
        return

    formatted_date = clone_date.strftime('%d-%b-%Y').upper()
    print(f"[+] Instance Clone Date Detected: {formatted_date}")

    # Step 2: Optimized query with ROWNUM cap to avoid full table scans on large EBS tables
    errors_query = """
    SELECT request_id,
           application_id,
           ledger_id,
           event_id,
           encoded_msg,
           creation_date
      FROM (
          SELECT xae.request_id,
                 xae.application_id,
                 xae.ledger_id,
                 xae.event_id,
                 xae.encoded_msg,
                 xae.creation_date
            FROM apps.xla_accounting_errors xae
           WHERE xae.creation_date >= :clone_dt
           ORDER BY xae.creation_date DESC
      )
     WHERE ROWNUM <= 100
    """

    try:
        print(f"[+] Querying top 100 SLA errors logged since clone date ({formatted_date})...")
        cursor.execute(errors_query, clone_dt=clone_date)
        rows = cursor.fetchall()

        if not rows:
            print(f"[-] No accounting errors found in SLA on/after clone date {formatted_date}.")
        else:
            print(f"[+] Total records retrieved: {len(rows)}\n")
            print("-" * 120)
            print(f"{'REQ_ID':<10} | {'APP_ID':<8} | {'LEDGER_ID':<10} | {'EVENT_ID':<12} | {'CREATION_DATE':<20} | {'ENCODED_MSG'}")
            print("-" * 120)

            for row in rows:
                req_id, app_id, l_id, evt_id, msg, dt = row
                dt_str = dt.strftime('%d-%b-%Y %H:%M:%S') if dt else 'N/A'
                clean_msg = str(msg).replace('\n', ' ').replace('\r', '') if msg else ''
                short_msg = (clean_msg[:40] + "...") if len(clean_msg) > 40 else clean_msg
                
                print(
                    f"{str(req_id):<10} | {str(app_id):<8} | {str(l_id):<10} | {str(evt_id):<12} | {dt_str:<20} | {short_msg}"
                )

            print("-" * 120)

    except Exception as exc:
        print(f"[-] Query Execution Error: {exc}")
    finally:
        cursor.close()
        connection.close()
        print("\n[+] Database connection closed.")


if __name__ == "__main__":
    fetch_sla_errors_fast()

[+] Oracle Client initialized in Thick mode from: C:\Users\E1445612\oracle64client

 ULTRA-FAST FETCH: SLA ERRORS FOR INSTANCE CLONE DATE
[+] Connecting to Oracle EBS Database...
[+] Connected successfully to EBS Instance.

[+] Checking Instance Clone Date from v$database...
[+] Instance Clone Date Detected: 08-SEP-2026
[+] Querying top 100 SLA errors logged since clone date (08-SEP-2026)...
[+] Total records retrieved: 100

------------------------------------------------------------------------------------------------------------------------
REQ_ID     | APP_ID   | LEDGER_ID  | EVENT_ID     | CREATION_DATE        | ENCODED_MSG
------------------------------------------------------------------------------------------------------------------------
570946123  | 200      | None       | None         | 08-Sep-2026 17:15:22 | There are no eligible transactions to ac...
570944051  | 200      | None       | None         | 08-Sep-2026 17:12:49 | There are no eligible transactions to ac...
5709